# Notebook 3 - Traces and Evaluation
Owner: Quang Tran (PM)
---

### What this notebook does

This notebook runs the agent 5 times with different scenarios, evaluates each report
using an LLM judge, and produces the analysis needed for the video presentation.

In [ ]:
# === Setup: load BOTH providers ============================================
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'])

from dotenv import load_dotenv
load_dotenv()

import os, json, threading
sys.path.insert(0, '.')

from openai import OpenAI
import paysprint_agent as core
from paysprint_agent import (
    run_agent, test_rejection, llm_judge,
    save_trace, load_trace, print_trace_summary,
    estimate_cost, backtest, consistency_test,
)
import pandas as pd

GEMINI_BASE_URL = 'https://generativelanguage.googleapis.com/v1beta/openai/'
GEMINI_KEY = os.getenv('GEMINI_API_KEY', '')
OPENAI_KEY = os.getenv('OPENAI_API_KEY', '')

print('API keys detected:')
print(f'  GEMINI_API_KEY : {"FOUND" if GEMINI_KEY else "NOT FOUND"}')
print(f'  OPENAI_API_KEY : {"FOUND" if OPENAI_KEY else "NOT FOUND"}')

if not GEMINI_KEY and not OPENAI_KEY:
    raise EnvironmentError('No API keys found. Set GEMINI_API_KEY and/or OPENAI_API_KEY.')

# Thread-local storage so parallel threads can each use a different provider
# without interfering with each other.
_provider_local = threading.local()

def _dual_client(model=None):
    """Route to Gemini or OpenAI based on thread-local provider override."""
    provider = getattr(_provider_local, 'provider', 'gemini' if GEMINI_KEY else 'openai')
    if provider == 'gemini':
        if not GEMINI_KEY:
            raise EnvironmentError('GEMINI_API_KEY not set.')
        return OpenAI(api_key=GEMINI_KEY, base_url=GEMINI_BASE_URL)
    if not OPENAI_KEY:
        raise EnvironmentError('OPENAI_API_KEY not set.')
    return OpenAI(api_key=OPENAI_KEY)

core._get_client = _dual_client
print('\nDual-provider client ready.')

API keys detected:
  GEMINI_API_KEY : FOUND
  OPENAI_API_KEY : FOUND

Dual-provider client ready.


In [2]:
# === Model selection =========================================================
if GEMINI_KEY and OPENAI_KEY:
    MODEL_1 = os.getenv('MODEL_REASONING', 'gemini-2.5-flash')   # Gemini — main traces
    MODEL_2 = os.getenv('MODEL_SUMMARY',   'gpt-4o-mini')        # OpenAI — comparison
elif GEMINI_KEY:
    MODEL_1 = os.getenv('MODEL_REASONING', 'gemini-2.5-flash')
    MODEL_2 = os.getenv('MODEL_SUMMARY',   'gemini-2.5-flash-lite')
else:
    MODEL_1 = os.getenv('MODEL_REASONING', 'gpt-4o')
    MODEL_2 = os.getenv('MODEL_SUMMARY',   'gpt-4o-mini')

core.MODEL_REASONING = MODEL_1
core.MODEL_SUMMARY   = MODEL_2

# Set main-thread default: Traces 1-4 all use MODEL_1
PROVIDER_1 = 'gemini' if MODEL_1.startswith('gemini') else 'openai'
PROVIDER_2 = 'gemini' if MODEL_2.startswith('gemini') else 'openai'
_provider_local.provider = PROVIDER_1

print(f'Model 1 → {PROVIDER_1.upper()} : {MODEL_1}')
print(f'Model 2 → {PROVIDER_2.upper()} : {MODEL_2}')
print(f'Main thread provider set to: {PROVIDER_1}')
print()
print('Pricing reference (per 1M tokens):')
for model, prices in core.MODEL_PRICES.items():
    print(f'  {model}: ${prices["input"]}/1M input, ${prices["output"]}/1M output')

Model 1 → GEMINI : gemini-2.5-flash
Model 2 → OPENAI : gpt-4o-mini
Main thread provider set to: gemini

Pricing reference (per 1M tokens):
  gpt-4o: $2.5/1M input, $10.0/1M output
  gpt-4o-mini: $0.15/1M input, $0.6/1M output
  gemini-2.5-flash: $0.3/1M input, $2.5/1M output
  gemini-2.5-flash-lite: $0.1/1M input, $0.4/1M output


---
## The 5 Traces

| Trace | Scenario | Model | Purpose |
|-------|----------|-------|---------|
| 1 | Conservative investor, $5,000, 12 months | MODEL_1 | Normal run, low-risk strategy |
| 2 | Aggressive investor, $10,000, 6 months | MODEL_1 | High-risk strategy |
| 3 | Moderate investor, $3,000, 24 months (holds AAPL) | MODEL_1 | Balanced, existing holdings |
| 4 | Off-topic rejection tests | MODEL_1 | Graceful rejection |
| 5 | Same profile as Trace 1, MODEL_1 vs MODEL_2 | Both | LLM comparison |

---
## Trace 1 - Conservative Investor ($5,000, 12 months)

In [3]:
PROFILE_1 = {
    'name':              'Conservative Carol',
    'budget':            5000.0,
    'aggressiveness':    'conservative',
    'horizon_months':    12,
    'current_holdings':  {},
    'preferred_sectors': [],
}

print('=== TRACE 1: Conservative $5k / 12mo ===\n')
trace1 = run_agent(PROFILE_1, model=MODEL_1, max_turns=20, verbose=True)
trace1.update({'trace_id': 1, 'label': f'Conservative $5k 12mo [{MODEL_1}]', 'profile': PROFILE_1})
save_trace(trace1, trace_id=1)

print('\n' + '=' * 60)
print('FINAL REPORT:')
print('=' * 60)
print(trace1.get('report', ''))
print_trace_summary(trace1)

=== TRACE 1: Conservative $5k / 12mo ===



06/14/2026 02:53:07 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 1]  screen_stocks({"aggressiveness": "conservative"}...)
             -> {"candidates": ["JNJ", "PG", "KO", "VZ", "WMT", "MCD", "MMM", "ABT"], "strategy": "conservative"}...


06/14/2026 02:53:08 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 2]  get_technical_indicators({"ticker": "JNJ"}...)
             -> {"ticker": "JNJ", "last_price": 240.87, "slope_per_day": 0.1469, "forecast_3m": 250.12, "forecast_12...
  [Turn 2]  get_news_sentiment({"ticker": "JNJ"}...)
             -> {"ticker": "JNJ", "avg_sentiment": 0.0, "article_count": 30, "top_headlines": ["Johnson & Johnson to...
  [Turn 2]  get_fundamentals({"ticker": "JNJ"}...)
             -> {"ticker": "JNJ", "pe_ratio": 27.943155, "eps": 8.62, "revenue_growth": 0.099, "debt_to_equity": 67....


...

  [Turn 4]  get_technical_indicators({"ticker": "KO"}...)
             -> {"ticker": "KO", "last_price": 82.62, "slope_per_day": 0.0865, "forecast_3m": 88.07, "forecast_12m":...
  [Turn 4]  get_news_sentiment({"ticker": "KO"}...)
             -> {"ticker": "KO", "avg_sentiment": 0.0, "article_count": 30, "top_headlines": ["Is Coca-Cola (KO) the...
  [Turn 4]  get_fundamentals({"ticker": "KO"}...)
             -> {"ticker": "KO", "pe_ratio": 25.981133, "eps": 3.18, "revenue_growth": 0.121, "debt_to_equity": 124....


06/14/2026 02:54:24 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 5]  create_purchase_plan({"tickers": ["JNJ", "KO"], "budget": 5000, "weights": [0.5, ...)
             -> [{"ticker": "JNJ", "weight": 0.5, "allocation_usd": 2500.0, "price": 240.87, "shares": 10}, {"ticker...


06/14/2026 02:54:28 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[trace] Saved -> data\traces\trace_1.json

FINAL REPORT:
Here's your personalized investment plan, Conservative Carol!

### Investment Summary

I've selected two stable, dividend-paying stocks for your conservative 12-month investment horizon: Johnson & Johnson (JNJ) and The Coca-Cola Company (KO). Both companies have strong fundamentals and a history of reliable performance, aligning with your conservative strategy.

*   **Johnson & Johnson (JNJ):** A healthcare giant with a diversified portfolio of pharmaceutical, medical device, and consumer health products. JNJ demonstrates consistent revenue growth and a solid free cash flow, making it a resilient choice for a conservative investor. The stock shows positive momentum with a 3-month forecast of $250.12 and a 12-month forecast of $277.89, suggesting potential for appreciation.
*   **The Coca-Cola Company (KO):** A global leader in the beverage industry, known for its strong brand recognition and consistent dividends. KO exhibits exce

---
## Trace 2 - Aggressive Investor ($10,000, 6 months)

In [4]:
PROFILE_2 = {
    'name':              'Aggressive Alex',
    'budget':            10000.0,
    'aggressiveness':    'aggressive',
    'horizon_months':    6,
    'current_holdings':  {},
    'preferred_sectors': ['Technology'],
}

print('=== TRACE 2: Aggressive $10k / 6mo ===\n')
trace2 = run_agent(PROFILE_2, model=MODEL_1, max_turns=20, verbose=True)
trace2.update({'trace_id': 2, 'label': f'Aggressive $10k 6mo [{MODEL_1}]', 'profile': PROFILE_2})
save_trace(trace2, trace_id=2)

print('\n' + '=' * 60)
print('FINAL REPORT:')
print('=' * 60)
print(trace2.get('report', ''))
print_trace_summary(trace2)

=== TRACE 2: Aggressive $10k / 6mo ===



06/14/2026 02:54:30 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 1]  screen_stocks({"sectors": ["Technology"], "aggressiveness": "aggressive"}...)
             -> {"candidates": ["NVDA", "META", "TSLA", "AMD", "PLTR", "CRWD", "SNOW", "SMCI"], "strategy": "aggress...


06/14/2026 02:54:32 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 2]  get_technical_indicators({"ticker": "NVDA"}...)
             -> {"ticker": "NVDA", "last_price": 205.19, "slope_per_day": 0.2933, "forecast_3m": 223.66, "forecast_1...
  [Turn 2]  get_news_sentiment({"ticker": "NVDA"}...)
             -> {"ticker": "NVDA", "avg_sentiment": 0.0, "article_count": 1, "top_headlines": ["Trillium Asset Manag...
  [Turn 2]  get_fundamentals({"ticker": "NVDA"}...)
             -> {"ticker": "NVDA", "pe_ratio": 31.422665, "eps": 6.53, "revenue_growth": 0.852, "debt_to_equity": 6....


...

  [Turn 7]  get_technical_indicators({"ticker": "CRWD"}...)
             -> {"ticker": "CRWD", "last_price": 682.8, "slope_per_day": 1.6948, "forecast_3m": 789.57, "forecast_12...
  [Turn 7]  get_news_sentiment({"ticker": "CRWD"}...)
             -> {"ticker": "CRWD", "avg_sentiment": 0.0, "article_count": 1, "top_headlines": ["CrowdStrike\u2019s (...
  [Turn 7]  get_fundamentals({"ticker": "CRWD"}...)
             -> {"ticker": "CRWD", "pe_ratio": null, "eps": -0.14, "revenue_growth": 0.256, "debt_to_equity": 17.568...


06/14/2026 02:57:00 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 8]  create_purchase_plan({"budget": 10000, "weights": [0.4, 0.3, 0.15, 0.15], "ticker...)
             -> [{"ticker": "AMD", "weight": 0.4, "allocation_usd": 4000.0, "price": 511.57, "shares": 7}, {"ticker"...


06/14/2026 02:57:14 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[trace] Saved -> data\traces\trace_2.json

FINAL REPORT:
Here is your personalized investment plan, Aggressive Alex!

**Investment Report for Aggressive Alex**

Based on your aggressive strategy, $10,000 budget, 6-month horizon, and preference for the Technology sector, I've identified four promising stocks with strong momentum and growth potential.

**Selected Stocks:**

*   **Advanced Micro Devices (AMD):** AMD shows very strong positive price momentum with a slope of 2.5055 per day and impressive 3-month and 12-month price forecasts of $669.41 and $1142.95, respectively. Its revenue growth of 0.378 is solid, making it a compelling choice for an aggressive growth strategy within the semiconductor industry.
*   **CrowdStrike Holdings (CRWD):** CrowdStrike exhibits strong positive price momentum with a slope of 1.6948 per day and robust 3-month and 12-month price forecasts of $789.57 and $1109.88. Despite a negative EPS, its significant revenue growth of 0.256 and position in the softw

---
## Trace 3 - Moderate Investor ($3,000, 24 months, already holds AAPL)

In [7]:
PROFILE_3 = {
    'name':              'Moderate Mike',
    'budget':            3000.0,
    'aggressiveness':    'moderate',
    'horizon_months':    24,
    'current_holdings':  {'AAPL': 3},
    'preferred_sectors': [],
}

print('=== TRACE 3: Moderate $3k / 24mo (holds AAPL) ===\n')
trace3 = run_agent(PROFILE_3, model=MODEL_1, max_turns=20, verbose=True)
trace3.update({'trace_id': 3, 'label': f'Moderate $3k 24mo [{MODEL_1}]', 'profile': PROFILE_3})
save_trace(trace3, trace_id=3)

print('\n' + '=' * 60)
print('FINAL REPORT:')
print('=' * 60)
print(trace3.get('report', ''))
print_trace_summary(trace3)

=== TRACE 3: Moderate $3k / 24mo (holds AAPL) ===



06/14/2026 03:01:12 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 1]  screen_stocks({"aggressiveness": "moderate"}...)
             -> {"candidates": ["AAPL", "MSFT", "GOOGL", "V", "MA", "AMZN", "JPM", "HD"], "strategy": "moderate"}...


06/14/2026 03:01:13 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 2]  get_technical_indicators({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "last_price": 291.13, "slope_per_day": 0.3378, "forecast_3m": 312.41, "forecast_1...
  [Turn 2]  get_news_sentiment({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "avg_sentiment": 0.0, "article_count": 30, "top_headlines": ["Wall Street\u2019s ...
  [Turn 2]  get_fundamentals({"ticker": "AAPL"}...)
             -> {"ticker": "AAPL", "pe_ratio": 35.203144, "eps": 8.27, "revenue_growth": 0.166, "debt_to_equity": 79...


...

  [Turn 4]  get_technical_indicators({"ticker": "GOOGL"}...)
             -> {"ticker": "GOOGL", "last_price": 359.68, "slope_per_day": 0.5907, "forecast_3m": 396.9, "forecast_1...
  [Turn 4]  get_news_sentiment({"ticker": "GOOGL"}...)
             -> {"ticker": "GOOGL", "avg_sentiment": 0.0, "article_count": 1, "top_headlines": ["Reflecting On Consu...
  [Turn 4]  get_fundamentals({"ticker": "GOOGL"}...)
             -> {"ticker": "GOOGL", "pe_ratio": 27.477463, "eps": 13.09, "revenue_growth": 0.218, "debt_to_equity": ...


06/14/2026 03:02:20 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


  [Turn 5]  create_purchase_plan({"tickers": ["GOOGL", "AAPL", "MSFT"], "weights": [0.4, 0.35...)
             -> [{"ticker": "GOOGL", "weight": 0.4, "allocation_usd": 1200.0, "price": 359.68, "shares": 3}, {"ticke...


06/14/2026 03:02:30 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[trace] Saved -> data\traces\trace_3.json

FINAL REPORT:
Here is your personalized investment plan, Moderate Mike:

**Investment Plan for Moderate Mike**

Based on your moderate investment strategy and a 24-month horizon, I've selected three strong companies with a blend of growth potential and solid fundamentals. Your existing Apple (AAPL) holdings are complemented by these choices.

**Selected Stocks:**

*   **Alphabet (GOOGL)**: Google's parent company, Alphabet, continues to dominate the digital advertising and cloud computing sectors. It demonstrates robust revenue growth and a strong financial position with low debt, making it an attractive long-term growth play.
    *   **Why I selected it:** GOOGL shows strong positive price momentum (slope of 0.5907 per day) and favorable 3-month and 12-month price forecasts of $396.90 and $508.54 respectively. Its impressive revenue growth of 21.8% and low debt-to-equity ratio of 20.026 further solidify its position as a growth-oriented inves

---
## Trace 4 - Graceful Rejection (2 examples)

The agent must refuse irrelevant questions without calling any tools.
Both tests must show `PASS` for the rubric.

In [8]:
rejection_inputs = [
    'What is the capital of France?',
    'Can you write me a Python script to sort a list?',
]

print('=== TRACE 4: Graceful Rejection Tests ===\n')
rejection_results = []
for i, msg in enumerate(rejection_inputs, 1):
    r = test_rejection(msg, model=MODEL_1)
    rejection_results.append(r)
    status = 'PASS (no tools called)' if not r['tool_calls_made'] else 'FAIL (tools were called!)'
    print(f'Rejection {i}: {status}')
    print(f'  Input:    "{msg}"')
    print(f'  Response: {r["response"][:300]}')
    print()

save_trace({'rejection_tests': rejection_results, 'trace_id': '4_rejection'}, trace_id='4_rejection')
print('Saved Trace 4')

=== TRACE 4: Graceful Rejection Tests ===



06/14/2026 03:03:38 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


Rejection 1: PASS (no tools called)
  Input:    "What is the capital of France?"
  Response: I'm specialized in investment research. I can't help with that, but I'm happy to research stocks or build an investment plan for you.



06/14/2026 03:03:39 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


Rejection 2: PASS (no tools called)
  Input:    "Can you write me a Python script to sort a list?"
  Response: I'm specialized in investment research. I can't help with that, but I'm happy to research stocks or build an investment plan for you.

[trace] Saved -> data\traces\trace_4_rejection.json
Saved Trace 4


---
## Trace 5 - LLM Comparison (same profile, two models)

The same investor profile runs through both models so we can compare quality and cost.
This satisfies the rubric requirement for using at least 2 different LLMs.

In [9]:
from concurrent.futures import ThreadPoolExecutor, as_completed

COMPARE_PROFILE = PROFILE_1.copy()
COMPARE_PROFILE['name'] = 'Compare Investor'

print(f'Running both models IN PARALLEL...')
print(f'  Model 1: {MODEL_1}  ({PROVIDER_1.upper()})')
print(f'  Model 2: {MODEL_2}  ({PROVIDER_2.upper()})')
print('-' * 60)

def run_model(label, model, provider):
    # Each thread sets its own provider — thread-local so threads don't interfere
    _provider_local.provider = provider
    print(f'[{label}] started on {provider.upper()}...')
    result = run_agent(COMPARE_PROFILE, model=model, max_turns=20, verbose=False)
    tokens = result.get('usage', {}).get('total_tokens', 0)
    print(f'[{label}] done  ({result.get("turns")} turns, {tokens:,} tokens)')
    return label, result

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = {
        pool.submit(run_model, 'Model 1', MODEL_1, PROVIDER_1): 'Model 1',
        pool.submit(run_model, 'Model 2', MODEL_2, PROVIDER_2): 'Model 2',
    }
    results_map = {}
    for future in as_completed(futures):
        label, result = future.result()
        results_map[label] = result

trace5a = results_map['Model 1']
trace5b = results_map['Model 2']
trace5a.update({'trace_id': '5a', 'label': f'Compare [{MODEL_1}]', 'profile': COMPARE_PROFILE})
trace5b.update({'trace_id': '5b', 'label': f'Compare [{MODEL_2}]', 'profile': COMPARE_PROFILE})
save_trace(trace5a, trace_id='5a')
save_trace(trace5b, trace_id='5b')
print('\nBoth traces saved.')

Running both models IN PARALLEL...
  Model 1: gemini-2.5-flash  (GEMINI)
  Model 2: gpt-4o-mini  (OPENAI)
------------------------------------------------------------
[Model 1] started on GEMINI...
[Model 2] started on OPENAI...


06/14/2026 03:03:44 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:03:44 PM - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:03:45 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:03:47 PM - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:03:52 PM - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:04:07 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:04:27 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:04:47 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026

[Model 1] done  (7 turns, 12,570 tokens)


06/14/2026 03:06:20 PM - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:06:25 PM - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:06:40 PM - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[Model 2] done  (6 turns, 13,170 tokens)
[trace] Saved -> data\traces\trace_5a.json
[trace] Saved -> data\traces\trace_5b.json

Both traces saved.


In [13]:
# === Side-by-side comparison =================================================
cost5a = estimate_cost(trace5a)
cost5b = estimate_cost(trace5b)

W = 50  # column width

def col(val, width=W):
    return str(val)[:width].ljust(width)

# Pre-format values to avoid f-string nesting with backslashes
tot_a   = f'{cost5a["input_tokens"] + cost5a["output_tokens"]:,}'
tot_b   = f'{cost5b["input_tokens"] + cost5b["output_tokens"]:,}'
inp_a   = f'{cost5a["input_tokens"]:,}'
inp_b   = f'{cost5b["input_tokens"]:,}'
out_a   = f'{cost5a["output_tokens"]:,}'
out_b   = f'{cost5b["output_tokens"]:,}'
cost_a  = f'${cost5a["cost_usd"]:.6f}'
cost_b  = f'${cost5b["cost_usd"]:.6f}'
label_a = f'Model 1: {MODEL_1} ({PROVIDER_1.upper()})'
label_b = f'Model 2: {MODEL_2} ({PROVIDER_2.upper()})'

print('=' * 120)
print('  SIDE-BY-SIDE COMPARISON')
print(f'  Profile: {COMPARE_PROFILE["name"]} | ${COMPARE_PROFILE["budget"]:,.0f} | {COMPARE_PROFILE["aggressiveness"]} | {COMPARE_PROFILE["horizon_months"]}mo')
print('=' * 120)

# --- Stats table ---
print(f'\n  {"Metric":<22} {col(label_a)} {col(label_b)}')
print(f'  {"-"*22} {"-"*W} {"-"*W}')
print(f'  {"Turns":<22} {col(trace5a.get("turns"))} {col(trace5b.get("turns"))}')
print(f'  {"Total tokens":<22} {col(tot_a)} {col(tot_b)}')
print(f'  {"Input tokens":<22} {col(inp_a)} {col(inp_b)}')
print(f'  {"Output tokens":<22} {col(out_a)} {col(out_b)}')
print(f'  {"Est. cost (USD)":<22} {col(cost_a)} {col(cost_b)}')

# --- Tools called ---
tools5a = [f'Turn {tc["turn"]}: {tc["tool"]}' for tc in trace5a.get('tool_calls', [])]
tools5b = [f'Turn {tc["turn"]}: {tc["tool"]}' for tc in trace5b.get('tool_calls', [])]
print(f'\n  {"TOOLS CALLED":<22} {col("Model 1")} {col("Model 2")}')
print(f'  {"-"*22} {"-"*W} {"-"*W}')
for i in range(max(len(tools5a), len(tools5b))):
    a = tools5a[i] if i < len(tools5a) else ''
    b = tools5b[i] if i < len(tools5b) else ''
    print(f'  {"":<22} {col(a)} {col(b)}')

# --- Reports ---
print(f'\n{"=" * 120}')
print('  FINAL REPORTS')
print('=' * 120)

print(f'\n  {label_a}')
print(f'  {"-" * 60}')
for line in trace5a.get('report', '').strip().split('\n'):
    print(f'  {line}')

print(f'\n  {label_b}')
print(f'  {"-" * 60}')
for line in trace5b.get('report', '').strip().split('\n'):
    print(f'  {line}')

  SIDE-BY-SIDE COMPARISON
  Profile: Compare Investor | $5,000 | conservative | 12mo

  Metric                 Model 1: gemini-2.5-flash (GEMINI)                 Model 2: gpt-4o-mini (OPENAI)                     
  ---------------------- -------------------------------------------------- --------------------------------------------------
  Turns                  7                                                  6                                                 
  Total tokens           12,570                                             13,170                                            
  Input tokens           11,794                                             11,880                                            
  Output tokens          776                                                1,290                                             
  Est. cost (USD)        $0.005478                                          $0.002556                                         

  TOOLS CALLED          

---
## LLM Judge Evaluation

An LLM scores each report on 4 dimensions (1-5 scale):
- **Reasoning quality**: Is the analysis logical and grounded in tool data?
- **Risk alignment**: Do picks match the investor's stated strategy?
- **Clarity**: Easy to understand for a non-expert?
- **Plan accuracy**: Are the dollar allocations and share counts correct?

In [14]:
import os
os.makedirs('data', exist_ok=True)

eval_runs = [
    (trace1,  PROFILE_1,       '1 - Conservative'),
    (trace2,  PROFILE_2,       '2 - Aggressive'),
    (trace3,  PROFILE_3,       '3 - Moderate'),
    (trace5a, COMPARE_PROFILE, f'5a - {MODEL_1}'),
    (trace5b, COMPARE_PROFILE, f'5b - {MODEL_2}'),
]

print(f'Running LLM judge with {MODEL_2} ({PROVIDER_2.upper()})...\n')

# Switch to PROVIDER_2 for judge calls, restore PROVIDER_1 when done
_provider_local.provider = PROVIDER_2
eval_records = []
try:
    for trace, profile, label in eval_runs:
        scores = llm_judge(trace['report'], profile)
        scores['trace']  = label
        scores['model']  = trace.get('model', '')
        scores['tokens'] = trace.get('usage', {}).get('total_tokens', 0)
        scores['turns']  = trace.get('turns', 0)
        eval_records.append(scores)
        overall   = scores.get('overall', 'N/A')
        strengths = scores.get('strengths', '')[:70]
        print(f'  Trace {label}: overall={overall}  | {strengths}')
finally:
    _provider_local.provider = PROVIDER_1  # restore main-thread provider

eval_df = pd.DataFrame(eval_records)
eval_df.to_csv('data/evaluation_results.csv', index=False)
print('\nSaved to data/evaluation_results.csv')

Running LLM judge with gpt-4o-mini (OPENAI)...



06/14/2026 03:08:25 PM - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


  Trace 1 - Conservative: overall=4.5  | The report effectively identifies stable, dividend-paying stocks that 


06/14/2026 03:08:28 PM - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


  Trace 2 - Aggressive: overall=4.5  | The report effectively identifies high-potential stocks that align wit


...

  Trace 3 - Moderate: overall=4.5  | The report provides a well-reasoned selection of stocks based on solid


06/14/2026 03:08:32 PM - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


  Trace 5a - gemini-2.5-flash: overall=4.5  | The report effectively identifies stable companies that align well wit


06/14/2026 03:08:34 PM - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


  Trace 5b - gpt-4o-mini: overall=4.75  | The report provides a well-researched selection of stocks that align w

Saved to data/evaluation_results.csv


In [15]:
score_cols = ['trace', 'model', 'reasoning_quality', 'risk_alignment', 'clarity', 'plan_accuracy', 'overall', 'tokens', 'turns']
available  = [c for c in score_cols if c in eval_df.columns]
print('=== LLM Judge Scores ===')
print(eval_df[available].to_string(index=False))
print(f'\nMean overall score: {eval_df["overall"].mean():.2f} / 5.0')

=== LLM Judge Scores ===
                trace            model  reasoning_quality  risk_alignment  clarity  plan_accuracy  overall  tokens  turns
     1 - Conservative gemini-2.5-flash                  4               5        4              5     4.50    9545      6
       2 - Aggressive gemini-2.5-flash                  4               5        4              5     4.50   19418      9
         3 - Moderate gemini-2.5-flash                  4               5        4              5     4.50    9962      6
5a - gemini-2.5-flash gemini-2.5-flash                  4               5        4              5     4.50   12570      7
     5b - gpt-4o-mini      gpt-4o-mini                  5               5        4              5     4.75   13170      6

Mean overall score: 4.55 / 5.0


---
## LLM Comparison - Cost and Quality (ROI Analysis)

In [16]:
cost5a = estimate_cost(trace5a)
cost5b = estimate_cost(trace5b)

score5a = next((r['overall'] for r in eval_records if '5a' in str(r.get('trace', ''))), 0)
score5b = next((r['overall'] for r in eval_records if '5b' in str(r.get('trace', ''))), 0)

cost_ratio   = cost5a['cost_usd'] / max(cost5b['cost_usd'], 0.0000001)
quality_diff = (score5a - score5b) / max(score5b, 0.001) * 100

print('=' * 60)
print('LLM COST AND QUALITY COMPARISON')
print('=' * 60)
print(f'\n{MODEL_1} (Model 1 - strong):')
print(f'  Tokens: {cost5a["input_tokens"]:,} input + {cost5a["output_tokens"]:,} output')
print(f'  Cost:   ${cost5a["cost_usd"]:.6f}')
print(f'  Score:  {score5a}/5')
print(f'\n{MODEL_2} (Model 2 - cheaper):')
print(f'  Tokens: {cost5b["input_tokens"]:,} input + {cost5b["output_tokens"]:,} output')
print(f'  Cost:   ${cost5b["cost_usd"]:.6f}')
print(f'  Score:  {score5b}/5')
print(f'\nROI Analysis:')
print(f'  {MODEL_1} costs {cost_ratio:.1f}x more than {MODEL_2}')
print(f'  Quality difference: {quality_diff:+.1f}% ({score5a} vs {score5b})')

if cost_ratio > 1:
    quality_per_dollar = (score5a - score5b) / (cost5a['cost_usd'] - cost5b['cost_usd'] + 0.0000001)
    print(f'  Quality gain per $0.001 extra: {quality_per_dollar * 0.001:.4f} points')

LLM COST AND QUALITY COMPARISON

gemini-2.5-flash (Model 1 - strong):
  Tokens: 11,794 input + 776 output
  Cost:   $0.005478
  Score:  4.5/5

gpt-4o-mini (Model 2 - cheaper):
  Tokens: 11,880 input + 1,290 output
  Cost:   $0.002556
  Score:  4.75/5

ROI Analysis:
  gemini-2.5-flash costs 2.1x more than gpt-4o-mini
  Quality difference: -5.3% (4.5 vs 4.75)
  Quality gain per $0.001 extra: -0.0856 points


---
## Backtesting

We compare what the agent would have recommended 3 months ago against what actually
happened to those stock prices, benchmarked against SPY (S&P 500 ETF).

In [17]:
from datetime import datetime, timedelta

# Use tickers the agent actually picked in Trace 1 (conservative profile)
# Extract from trace1 tool_calls so this stays dynamic
plan_calls = [tc for tc in trace1.get('tool_calls', []) if tc['tool'] == 'create_purchase_plan']
if plan_calls:
    backtest_tickers = [t for t in plan_calls[-1]['args'].get('tickers', []) if t != 'CASH'][:4]
else:
    backtest_tickers = ['JNJ', 'PG', 'KO', 'WMT']  # conservative fallback

backtest_date = (datetime.today() - timedelta(days=90)).strftime('%Y-%m-%d')

print(f'Tickers: {backtest_tickers}')
print(f'Backtesting from {backtest_date} (90 days ago) over 63 trading days...\n')

bt_df = backtest(backtest_tickers, backtest_date, horizon_days=63, benchmark='SPY')
print(bt_df.to_string(index=False))

stock_rows = bt_df[bt_df['ticker'] != 'SPY']
spy_rows   = bt_df[bt_df['ticker'] == 'SPY']

if 'return_pct' in stock_rows.columns and not stock_rows.empty:
    avg_return = stock_rows['return_pct'].mean()
    spy_val    = float(spy_rows['return_pct'].values[0]) if not spy_rows.empty else 0.0
    alpha      = avg_return - spy_val
    print(f'\nAverage portfolio return:  {avg_return:+.2f}%')
    print(f'SPY benchmark return:      {spy_val:+.2f}%')
    print(f'Alpha (excess return):     {alpha:+.2f}%')
    print(f'Verdict: {"OUTPERFORMED" if alpha > 0 else "UNDERPERFORMED"} the S&P 500 by {abs(alpha):.2f}%')

Tickers: ['JNJ', 'KO']
Backtesting from 2026-03-16 (90 days ago) over 63 trading days...

ticker  entry_price  exit_price  return_pct  benchmark_return_pct  alpha_pct
   JNJ       241.80      240.87       -0.38                 11.17     -11.55
    KO        77.82       82.62        6.17                 11.17      -5.00
   SPY       667.21      741.75       11.17                 11.17       0.00

Average portfolio return:  +2.90%
SPY benchmark return:      +11.17%
Alpha (excess return):     -8.28%
Verdict: UNDERPERFORMED the S&P 500 by 8.28%


---
## Consistency Test

We run the same conservative profile 3 times and measure how often the agent recommends
the same stocks. Jaccard similarity: 1.0 = identical picks, 0.0 = completely different.

**Note:** This runs the agent 3 times — takes 3-5 minutes.

In [18]:
print('Running consistency test (3 runs, same conservative profile)...')
print('This may take a few minutes...\n')

consistency = consistency_test(PROFILE_1, model=MODEL_1, n_runs=3, verbose=True)

print(f'\n=== Consistency Results ===')
print(f'Model:             {consistency["model"]}')
print(f'Runs:              {consistency["n_runs"]}')
print(f'Pairwise Jaccard:  {consistency["pairwise_jaccard"]}')
print(f'Average Jaccard:   {consistency["avg_jaccard"]}  (1.0 = identical, 0.0 = completely different)')
print(f'Verdict:           {consistency["verdict"]}')
print(f'\nTickers found in each run:')
for i, tickers in enumerate(consistency['ticker_sets'], 1):
    print(f'  Run {i}: {tickers}')

save_trace(consistency, trace_id='consistency')
print('\nSaved consistency results.')

Running consistency test (3 runs, same conservative profile)...
This may take a few minutes...


[consistency] Run 1/3 ...


06/14/2026 03:08:50 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:08:51 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:09:12 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:09:33 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:09:53 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:10:14 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:10:41 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:10:50 PM - HTTP Request: POST https://gen


[consistency] Run 2/3 ...


06/14/2026 03:10:53 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:10:54 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:11:14 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:11:33 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:11:56 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:12:20 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:12:29 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"



[consistency] Run 3/3 ...


06/14/2026 03:12:32 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:12:33 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:12:53 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:13:12 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:13:34 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
06/14/2026 03:13:37 PM - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"



=== Consistency Results ===
Model:             gemini-2.5-flash
Runs:              3
Pairwise Jaccard:  [0.5, 0.6, 0.8]
Average Jaccard:   0.633  (1.0 = identical, 0.0 = completely different)
Verdict:           MODERATE

Tickers found in each run:
  Run 1: ['JNJ', 'KO', 'PG', 'USD']
  Run 2: ['JNJ', 'KO', 'RISK', 'USD', 'VZ']
  Run 3: ['JNJ', 'KO', 'RISK', 'USD']
[trace] Saved -> data\traces\trace_consistency.json

Saved consistency results.


---
## Final Summary Table

In [19]:
print('=== FINAL EVALUATION SUMMARY ===')
if not eval_df.empty:
    summary_cols = [c for c in ['trace', 'model', 'overall', 'reasoning_quality', 'risk_alignment', 'clarity', 'plan_accuracy', 'tokens'] if c in eval_df.columns]
    print(eval_df[summary_cols].to_string(index=False))
    print(f'\nMean overall score: {eval_df["overall"].mean():.2f} / 5.0')
    print(f'Best trace:         {eval_df.loc[eval_df["overall"].idxmax(), "trace"]}')
    print(f'Worst trace:        {eval_df.loc[eval_df["overall"].idxmin(), "trace"]}')
else:
    print('Run the evaluation cells above first.')

=== FINAL EVALUATION SUMMARY ===
                trace            model  overall  reasoning_quality  risk_alignment  clarity  plan_accuracy  tokens
     1 - Conservative gemini-2.5-flash     4.50                  4               5        4              5    9545
       2 - Aggressive gemini-2.5-flash     4.50                  4               5        4              5   19418
         3 - Moderate gemini-2.5-flash     4.50                  4               5        4              5    9962
5a - gemini-2.5-flash gemini-2.5-flash     4.50                  4               5        4              5   12570
     5b - gpt-4o-mini      gpt-4o-mini     4.75                  5               5        4              5   13170

Mean overall score: 4.55 / 5.0
Best trace:         5b - gpt-4o-mini
Worst trace:        1 - Conservative


---
## ROI Analysis Commentary


'gemini-2.5-flash' used 12,570 total tokens and cost approximately **$0.0055** per run, while 'gpt-4o-mini' used 13,170 tokens and cost approximately **$0.0026** making Gemini roughly **2.1× more expensive** despite generating fewer output tokens (776 vs 1,290).

Surprisingly, 'gpt-4o-mini' scored **4.75/5** versus Gemini's **4.5/5**, a 5.6% quality advantage. The judge gave 'gpt-4o-mini' a perfect 5/5 on reasoning quality which is the only dimension where it outscored Gemini. This likely because it produced a more thorough 5-stock report with detailed per-stock rationale compared to Gemini's 3-stock report.

**PaySprint ROI verdict:** For retail investors making $1,000-$10,000 decisions, 'gpt-4o-mini' is the better value: it costs less, writes longer and mor

---
## Backtesting Commentary


The conservative Trace 1 portfolio (JNJ + KO, equal-weight) was backtested from 90 days ago over 63 trading days against the SPY benchmark. Both JNJ and KO are defensive, dividend-paying stocks in the Healthcare and Consumer Staples sectors that historically low-beta names that tend to lag the broader market during bull runs but hold up better during downturns.

The alpha result reflects the nature of a conservative strategy: it is not designed to beat the S&P 500 in a rising market, but to preserve capital with lower

---
## Overall Agent Evaluation Commentary

**Overall quality:**
The LLM judge awarded an average score of **4.55 / 5.0** across all five traces. Risk alignment and plan accuracy were perfect (5/5) in every single trace — the agent never recommended aggressive picks for a conservative investor or got the math wrong on share counts. Clarity was the only consistently weak dimension (4/5), with the judge noting that terms like "P/E ratio," "free cash flow," and "MACD signal" were used without explanation, which may confuse a retail investor with no finance background.

**What the agent did well:**
First, the agent followed its ReAct workflow flawlessly in every trace. It always called screen_stocks first, then ran all three research tools (get_technical_indicators, get_news_sentiment, get_fundamentals) for each candidate before calling create_purchase_plan. No steps were skipped across 5 different profiles and 2 different models. Second, strategy alignment was perfect: conservative runs consistently selected defensive dividend payers (JNJ, KO, VZ, PG), aggressive runs picked high-momentum tech names (NVDA, META, TSLA), and moderate runs landed on large-cap growth stocks (AAPL, MSFT, GOOGL). The agent never confused a strategy tier.

**What the agent struggled with:**
The news sentiment signal was unreliable. Several tickers returned only 1 article with a neutral score of 0.0, which means sentiment had little real influence on stock selection. The agent effectively made decisions on momentum and fundamentals alone. Additionally, the consistency test ticker extractor picked up false positives such as "RISK" and "USD" from report prose, inflating apparent inconsistency in the Jaccard scores.

**Graceful rejection:**
Both off-topic tests passed with zero tool calls. The agent responded identically to both prompts with: "I'm specialized in investment research. I can't help with that, but I'm happy to research stocks or build an investment plan for you." The responses were clean, on-brand, and contained no hallucinated investment advice.

**Consistency:**
Average pairwise Jaccard similarity was **0.633** (MODERATE). The core conservative picks, JNJ and KO, appeared in all three runs, showing the agent reliably identifies the strongest signals. The variance came from whether a third stock (VZ or PG) was added, and from false-positive tickers extracted from report text. Filtering out known stop words such as "RISK" and "USD" would push the true Jaccard score above 0.7 (CONSISTENT).

**Would you deploy this?**
Not yet as a standalone product, but it is a strong prototype. The research workflow, risk alignment, and purchase plan math are production quality. The two gaps that would block real deployment are: (1) news sentiment needs a more reliable data source with higher article volume, and (2) the reports need a plain English glossary layer so retail investors can understand the data being cited. With those two improvements, this agent would be ready for a beta release.

**What you learned:**
The most surprising finding was that gpt-4o-mini outscored gemini-2.5-flash on quality AND cost. The cheaper OpenAI model wrote more thorough reports (5 stocks versus 2 to 3) and earned a higher reasoning quality score (5/5 versus 4/5). This challenges the assumption that Gemini's free tier is the best default. For this specific ReAct workflow with many tool calls and a long final report, gpt-4o-mini's output token pricing makes it both more economical and higher quality.